# 🚁 UAV Aerial Mosaicking — Classical Baseline (ORB / SIFT)

Notebook triển khai pipeline ghép ảnh cổ điển từ chuỗi keyframe UAV (NPU Drone-Map dataset):
1. **DataLoader:** Đọc sequence ảnh undistorted và metadata (`config.cfg`, `trajectory.txt`).
2. **Feature Extraction & Matching:** ORB / SIFT + FLANN / BFMatcher + Lowe's ratio test.
3. **Homography Estimation:** RANSAC tìm $H_{i \to i-1}$, lọc inliers, tính $H_{i \to 0}$ tích lũy.
4. **Warp & Stitch:** Chiếu các frame lên hệ toạ độ toàn cảnh (Global coordinate canvas).
5. **Evaluation Logger:** Lưu ma trận homography tích lũy và metric (Inliers, Inlier ratio, RMSE, latency) ra file JSON.


In [ ]:
import os
import sys
import time
import json
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")


In [ ]:
# Tự động xác định đường dẫn gốc dự án
REPO_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"

# Chọn sequence thử nghiệm (mặc định phantom3-village-kfs)
SEQUENCE_NAME = "phantom3-village-kfs"
SEQUENCE_PATH = DATA_DIR / SEQUENCE_NAME

OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / SEQUENCE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset Path: {SEQUENCE_PATH.resolve()}")
print(f"Output Path:  {OUTPUT_DIR.resolve()}")


In [ ]:
def load_sequence(seq_path: Path):
    """Đọc danh sách ảnh keyframe và metadata trajectory/config."""
    rgb_dir = seq_path / "rgb"
    if not rgb_dir.exists():
        sub_dirs = list(seq_path.glob("*/rgb"))
        if sub_dirs:
            rgb_dir = sub_dirs[0]

    # Sắp xếp ảnh theo timestamp
    img_files = sorted(
        list(rgb_dir.glob("*.jpg")) + list(rgb_dir.glob("*.png")),
        key=lambda p: float(p.stem) if p.stem.replace(".", "", 1).isdigit() else p.name
    )

    # Đọc trajectory ground truth nếu có
    traj_file = seq_path / "trajectory.txt"
    trajectories = {}
    if traj_file.exists():
        with open(traj_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 8:
                    trajectories[float(parts[0])] = [float(x) for x in parts[1:]]

    # Đọc config camera
    cfg_file = seq_path / "config.cfg"
    camera_cfg = {}
    if cfg_file.exists():
        with open(cfg_file, "r") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    camera_cfg[k.strip()] = v.strip()

    return img_files, trajectories, camera_cfg

img_files, trajectories, camera_cfg = load_sequence(SEQUENCE_PATH)
print(f"Tìm thấy: {len(img_files)} frames | {len(trajectories)} pose ground truth")
if img_files:
    sample = cv2.imread(str(img_files[0]))
    print(f"Frame resolution: {sample.shape[1]}x{sample.shape[0]} px")


### 2. Feature Extraction & Matching (ORB / SIFT)
Lọc matches bằng Lowe's ratio test để loại trừ các điểm tương đồng sai do nhiễu.


In [ ]:
def extract_and_match(img1, img2, method="SIFT", max_features=2000, ratio_thresh=0.75):
    """Trích xuất keypoints và match giữa 2 ảnh."""
    g1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY) if len(img1.shape) == 3 else img1
    g2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY) if len(img2.shape) == 3 else img2

    t0 = time.perf_counter()
    if method.upper() == "SIFT":
        detector = cv2.SIFT_create(nfeatures=max_features)
        kp1, des1 = detector.detectAndCompute(g1, None)
        kp2, des2 = detector.detectAndCompute(g2, None)
        matcher = cv2.BFMatcher(cv2.NORM_L2)
    elif method.upper() == "ORB":
        detector = cv2.ORB_create(nfeatures=max_features)
        kp1, des1 = detector.detectAndCompute(g1, None)
        kp2, des2 = detector.detectAndCompute(g2, None)
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    else:
        raise ValueError(f"Unknown method {method}")

    if des1 is None or des2 is None or len(des1) < 4 or len(des2) < 4:
        return [], kp1, kp2, 0.0

    matches = matcher.knnMatch(des1, des2, k=2)
    good = [m[0] for m in matches if len(m) == 2 and m[0].distance < ratio_thresh * m[1].distance]
    dur = time.perf_counter() - t0
    return good, kp1, kp2, dur

# Visual check trên 2 frame đầu
if len(img_files) >= 2:
    im0 = cv2.imread(str(img_files[0]))
    im1 = cv2.imread(str(img_files[1]))
    good_matches, kp0, kp1, dur = extract_and_match(im0, im1, method="SIFT")
    print(f"SIFT Matches: {len(good_matches)} điểm ({dur*1000:.1f} ms)")
    vis = cv2.drawMatches(im0, kp0, im1, kp1, good_matches[:60], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(12, 5))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"SIFT Top 60 Matches (Total: {len(good_matches)})")
    plt.axis("off")
    plt.show()


### 3. Ước lượng Homography & RANSAC
Ước lượng ma trận biến đổi phối cảnh $H_{i \to i-1}$ và tính ma trận tích lũy $H_{i \to 0} = H_{i-1 \to 0} \cdot H_{i \to i-1}$.


In [ ]:
def estimate_homography(kp_prev, kp_curr, matches, ransac_thresh=3.0):
    """Ước lượng H từ frame curr sang frame prev bằng RANSAC."""
    if len(matches) < 4:
        return None, np.array([]), 0, 0.0, float("inf")

    pts_prev = np.float32([kp_prev[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts_curr = np.float32([kp_curr[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(pts_curr, pts_prev, cv2.RANSAC, ransac_thresh)
    if H is None:
        return None, np.array([]), 0, 0.0, float("inf")

    inliers = mask.ravel().astype(bool)
    inlier_count = int(np.sum(inliers))
    inlier_ratio = inlier_count / len(matches) if len(matches) > 0 else 0.0

    if inlier_count > 0:
        p_curr_in = pts_curr[inliers]
        p_prev_in = pts_prev[inliers]
        p_proj = cv2.perspectiveTransform(p_curr_in, H)
        rmse = float(np.sqrt(np.mean(np.linalg.norm(p_prev_in - p_proj, axis=2)**2)))
    else:
        rmse = float("inf")

    return H, mask, inlier_count, inlier_ratio, rmse


### 4. Canvas Bounding Box & Stitching Pipeline
Tính canvas bao phủ toàn bộ quỹ đạo bay và warp từng frame về toạ độ mosaic.


In [ ]:
def compute_canvas(shapes, homographies):
    """Tính kích thước canvas toàn cục và ma trận tịnh tiến T."""
    all_pts = []
    for (h, w), H in zip(shapes, homographies):
        corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32).reshape(-1, 1, 2)
        warped = cv2.perspectiveTransform(corners, H)
        all_pts.append(warped.reshape(-1, 2))
    all_pts = np.vstack(all_pts)
    min_x, min_y = np.floor(all_pts.min(axis=0)).astype(int)
    max_x, max_y = np.ceil(all_pts.max(axis=0)).astype(int)
    w_canvas = max_x - min_x
    h_canvas = max_y - min_y
    T = np.array([[1.0, 0.0, -min_x], [0.0, 1.0, -min_y], [0.0, 0.0, 1.0]], dtype=np.float64)
    return w_canvas, h_canvas, T

def run_mosaic_pipeline(image_files, method="SIFT", num_frames=20, max_features=2000):
    """Thực thi ghép ảnh tuần tự và ghi metric."""
    n = min(len(image_files), num_frames)
    images = [cv2.imread(str(image_files[i])) for i in range(n)]
    shapes = [(img.shape[0], img.shape[1]) for img in images]

    H_accum = [np.eye(3, dtype=np.float64)]
    metrics = []
    print(f"Ghép {n} frames ({method})...")

    for i in range(1, n):
        t0 = time.perf_counter()
        good, kp_prev, kp_curr, _ = extract_and_match(images[i-1], images[i], method=method, max_features=max_features)
        H_rel, _, inliers, inlier_ratio, rmse = estimate_homography(kp_prev, kp_curr, good)
        dur = time.perf_counter() - t0

        if H_rel is None:
            print(f"Frame {i} RANSAC failed. Giữ nguyên pose.")
            H_accum.append(H_accum[-1].copy())
            continue

        H_global = H_accum[-1] @ H_rel
        H_accum.append(H_global)

        metrics.append({
            "frame": i,
            "timestamp": Path(image_files[i]).stem,
            "method": method,
            "matches": len(good),
            "inliers": inliers,
            "inlier_ratio": round(inlier_ratio, 4),
            "rmse": round(rmse, 4),
            "latency_ms": round(dur * 1000, 2),
            "H_accum": H_global.tolist()
        })
        if i % 5 == 0 or i == n - 1:
            print(f"  Frame {i:02d}/{n-1}: inliers={inliers:3d} ({inlier_ratio*100:.1f}%), RMSE={rmse:.2f}px, {dur*1000:.1f}ms")

    # Xây dựng canvas
    w_canv, h_canv, T = compute_canvas(shapes, H_accum)
    max_dim = 5000
    scale = min(1.0, max_dim / max(w_canv, h_canv))
    out_w, out_h = int(w_canv * scale), int(h_canv * scale)
    S = np.diag([scale, scale, 1.0])
    print(f"Canvas size: {out_w}x{out_h} px (scale: {scale:.3f})")

    mosaic = np.zeros((out_h, out_w, 3), dtype=np.uint8)
    weights = np.zeros((out_h, out_w), dtype=np.float32)

    for i in range(n):
        H_f = S @ T @ H_accum[i]
        warped = cv2.warpPerspective(images[i], H_f, (out_w, out_h))
        mask = (cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY) > 0).astype(np.float32)
        overlap = weights > 0
        new_px = (mask > 0) & (~overlap)
        blend_px = (mask > 0) & overlap
        mosaic[new_px] = warped[new_px]
        if np.any(blend_px):
            mosaic[blend_px] = (0.5 * mosaic[blend_px].astype(np.float32) + 0.5 * warped[blend_px].astype(np.float32)).astype(np.uint8)
        weights += mask

    return mosaic, H_accum, metrics


### 5. Chạy ghép 20 frames đầu & Xuất kết quả


In [ ]:
NUM_TEST_FRAMES = 20
mosaic_img, H_list, logs = run_mosaic_pipeline(img_files, method="SIFT", num_frames=NUM_TEST_FRAMES)

# Lưu ảnh mosaic kết quả
out_img_file = OUTPUT_DIR / f"mosaic_{SEQUENCE_NAME}_{NUM_TEST_FRAMES}frames.jpg"
cv2.imwrite(str(out_img_file), mosaic_img)
print(f"💾 Đã lưu ảnh: {out_img_file.resolve()}")

# Lưu log metrics
out_log_file = OUTPUT_DIR / f"metrics_{SEQUENCE_NAME}_{NUM_TEST_FRAMES}frames.json"
with open(out_log_file, "w", encoding="utf-8") as f:
    json.dump({"sequence": SEQUENCE_NAME, "frames": logs}, f, indent=2)
print(f"💾 Đã lưu metrics log: {out_log_file.resolve()}")

# Hiển thị trực quan
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(mosaic_img, cv2.COLOR_BGR2RGB))
plt.title(f"Mosaic Result: {SEQUENCE_NAME} ({NUM_TEST_FRAMES} frames)")
plt.axis("off")
plt.show()
